# The Hidden Layer — An Agentic AI Infiltration Exercise

Welcome to **The Hidden Layer**, a spy-themed agentic AI exercise!

You will build an LLM-powered agent that infiltrates a military base on the fictional **Isla Perdida**, talks to informants, gathers materials, crafts weapons, and collects **10 classified dossiers**.

## The Game
- **8x8 grid** with informants, facilities, safe houses, robots, jungle, and dossier caches
- **Operative starts** at (7,0) with 3 health, 0 dossiers, empty inventory
- **Win condition**: Collect 10 dossiers and survive (health > 0)
- **Limited visibility**: You can only see adjacent cells
- **15 total dossiers** available through caches, deliveries, and robot takedowns

## What You Implement
| TODO | File | Description |
|------|------|-------------|
| 1 | `agent.py` | `think_llm()` — LLM-powered decision function |

In [ ]:
# ── Setup (run this first) ──────────────────────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists("coding-exercises"):
        !cd coding-exercises && git pull
    else:
        !git clone -b week3 https://github.com/eth-bmai-fs26/coding-exercises.git
    os.chdir("coding-exercises/agentic_ai_spy")

sys.path.insert(0, ".")

!pip install google-genai --quiet
print("Setup complete ✓")

In [ ]:
from hidden_layer import *
print("The Hidden Layer loaded successfully!")

---
## Exploring the Base

Before implementing your agent, let's explore the game world and understand the tools available.

The operative interacts with the world through **9 tools**:
- `scan()` — Survey adjacent cells (free)
- `move(direction)` — Move north/south/east/west
- `codec(question)` — Talk to informants, safe house operators, facility engineers
- `collect()` — Pick up items in current cell
- `fabricate(item)` — Buy/craft at facilities
- `use(item)` — Use consumables (Field Rations, Med Kit)
- `engage()` — Fight a robot (need correct weapon!)
- `hide()` — Hide at safe house (costs 1 dossier, +1 health)
- `sitrep()` — Check operative state

In [ ]:
# Create a game and explore manually
operative, world, tools = create_game()
tools.set_oracle(stub_oracle)

# See the full map (for exploration only — the agent sees limited visibility!)
from hidden_layer.display import render_grid
print(render_grid(world, operative, show_all=True))

In [ ]:
# Try some tools manually
print("=== Scan ===")
print(tools.execute("scan", {}).message)

print("\n=== Move east (to Dr. Vapnik) ===")
print(tools.execute("move", {"direction": "east"}).message)

print("\n=== Codec with Dr. Vapnik ===")
print(tools.execute("codec", {"question": "What can you tell me about the base?"}).message)

print("\n=== Sitrep ===")
print(tools.execute("sitrep", {}).message)

### Understanding the Tool Call Format

Your think function must return a string in this format:
```
TOOL: tool_name(arg="value")
```

Examples:
```python
'TOOL: move(direction="north")'
'TOOL: collect()'
'TOOL: codec(question="Do you have a job for me?")'
'TOOL: fabricate(item="Flamethrower")'
'TOOL: engage()'
```

The `parse_tool_call()` function extracts the tool name and arguments. If parsing fails, it defaults to `scan()`.

In [ ]:
# Test the parser
from hidden_layer.agent import parse_tool_call

print(parse_tool_call('TOOL: move(direction="north")'))
print(parse_tool_call('TOOL: collect()'))
print(parse_tool_call('I should move east. TOOL: move(direction="east")'))
print(parse_tool_call('gibberish'))  # falls back to ('scan', {})

---
## API Key Setup

Set your Gemini API key. In Colab, use the **Secrets** panel (key icon in the left sidebar) to add `GEMINI_API_KEY`.

In [ ]:
import os
from google import genai

# Option 1: Set your API key directly
# os.environ["GEMINI_API_KEY"] = "your-key-here"

# Option 2: In Colab, use Secrets (recommended)
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    api_key = os.environ.get("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)
print("Gemini client ready!")

---
## TODO: LLM Think Function

Implement `think_llm()` in `agent.py` — the LLM decides the operative's next action each turn.

The function should:
1. **Build a system message** with:
   - Agent's role ("You are a spy operative, collect 10 dossiers to complete the mission")
   - Available tools (`TOOLS_DESCRIPTION`)
   - Instructions to output exactly one `TOOL: name(args)` call

2. **Build a user message** with:
   - Current operative status (`operative.status_text()`)
   - Recent history (last ~10-15 entries)
   - Key journal entries (past informant conversations)

3. **Call the API** and return the response text

```python
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=user_message,
    config=genai.types.GenerateContentConfig(
        system_instruction=system_message,
        max_output_tokens=500,
    ),
)
return response.text
```

### Available Information
```python
operative.position       # (row, col) tuple
operative.visited        # set of visited (row, col) tuples
operative.health         # current health (1-3)
operative.dossiers       # current dossiers
operative.inventory      # list of item name strings
operative.has_item(name) # check if operative has an item

world.get_cell(row, col)           # returns Cell with .cell_type, .items, .npc, etc.
world.get_adjacent(row, col)       # returns {"north": Cell, "south": Cell, ...}
world.is_passable(row, col)        # True if cell can be entered

CellType.CACHE, CellType.INFORMANT, CellType.FORGE, CellType.LAB, CellType.SAFEHOUSE, etc.
```

### Prompt Engineering Tips

A naive prompt will get the agent stuck in loops. Consider these techniques:

- **Ban wasted actions**: `scan()` runs automatically every turn — tell the LLM to never call it. Same for `sitrep()`.
- **Handle failed actions**: Tell the LLM explicitly: "If collect() fails, the cell is empty — leave immediately."
- **Encourage exploration**: Include `operative.visited` in the user message so the LLM knows which cells are unexplored. Remind it that the map is 8x8 and it must explore widely.
- **Stuck detection**: Count how many consecutive turns the agent has been at the same position. If stuck, inject a warning into the user message.
- **Dossier math**: 3 caches = 3 dossiers. That's not enough! The LLM needs to know it must talk to informants, do deliveries, and fight robots.

In [ ]:
# Run the game
from hidden_layer.oracle import llm_oracle

oracle_fn = lambda npc, q, o: llm_oracle(npc, q, o, client)

result = play_with_llm(
    think_fn=lambda operative, world, history: think_llm(operative, world, history, client),
    oracle_fn=oracle_fn,
    max_turns=75,
    delay=0.3,
)
print(f"\nResult: {'Mission Complete!' if result['won'] else 'Mission Failed...'} | Dossiers: {result['dossiers']} | Health: {result['health']} | Turns: {result['turns']}")
print(f"Game log saved to: {result['log_file']}")

In [ ]:
# Download the game log for debugging
try:
    from google.colab import files
    files.download(result["log_file"])
except ImportError:
    print(f"Log file: {result['log_file']}")
    print("(Open the file to inspect your agent's decisions turn by turn)")

---
## Reflection

After running your agent, consider:

1. **Did the agent collect enough dossiers?** If not, what went wrong?
2. **How did it handle informant clues?** Did it interpret cryptic dialogue correctly?
3. **Did it get stuck?** If so, what prompt engineering changes could fix it?
4. **What's the tradeoff** between a simple prompt and a sophisticated one? (cost, reliability, flexibility)

### The Big Insight

> *The perceive → think → act loop is the skeleton of every agentic AI system. The tools didn't change. The game didn't change. The only thing that matters is the quality of the `think` function — how well your LLM reasons about state, interprets language, and plans its next move. That's the hidden layer.*